In [ ]:
# 03 Terrain Processing

## Purpose
Add terrain attributes to the deployment-site database.

## Inputs
- Nigeria ADM0 boundary
- Copernicus DEM GLO-30 data downloaded in chunks
- Existing 5 km deployment-site grid
- `outputs/tables/nigeria_landcover_fraction_5km.csv`

## Main processing steps
- Download Copernicus DEM data for Nigeria in memory-safe chunks
- Merge DEM chunks into one national DEM mosaic
- Reproject DEM to a meter-based CRS
- Calculate mean elevation for each 5 km grid cell
- Derive slope from the DEM
- Calculate mean slope for each 5 km grid cell
- Merge terrain attributes into the land-cover database

## Outputs
- `data/processed/nigeria_dem_250m_mosaic.tif`
- `data/processed/nigeria_dem_250m_3857.tif`
- `data/processed/nigeria_slope_250m_3857.tif`
- `outputs/tables/deployment_site_database_v2.csv`

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt

print("Libraries loaded.")

Libraries loaded.


In [ ]:
boundary_path = Path("../data/raw/boundaries/geoBoundaries-NGA-ADM0_simplified.geojson")

nigeria = gpd.read_file(boundary_path)

print(nigeria.crs)
nigeria.head()

EPSG:4326


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,the Federal Republic of Nigeria,NGA,39225661B33378512450279,NGA,ADM0,"POLYGON ((5.67705 13.82055, 5.63161 13.83633, ..."


In [3]:
import rasterio
from pathlib import Path
from dem_stitcher.stitcher import stitch_dem

dem_dir = Path("../data/raw/copernicus_dem")
dem_dir.mkdir(parents=True, exist_ok=True)

# Tiny test area inside Nigeria: about 0.5 x 0.5 degrees
test_bounds = [7.0, 8.5, 7.5, 9.0]  # min lon, min lat, max lon, max lat

dem_array_test, dem_profile_test = stitch_dem(
    bounds=test_bounds,
    dem_name="glo_30",
    dst_ellipsoidal_height=False,
    dst_area_or_point="Area",
    dst_resolution=0.0025,
    n_threads_downloading=1,
    n_threads_reproj=1
)

test_dem_path = dem_dir / "test_copernicus_glo30_250m.tif"

with rasterio.open(test_dem_path, "w", **dem_profile_test) as dst:
    dst.write(dem_array_test, 1)

print(f"Test DEM saved to: {test_dem_path}")

Opening glo_30 Datasets:   0%|          | 0/1 [00:00<?, ?it/s]

Reading tile imagery: 100%|██████████| 1/1 [00:33<00:00, 33.69s/it]


Test DEM saved to: ..\data\raw\copernicus_dem\test_copernicus_glo30_250m.tif


In [5]:
from pathlib import Path
import geopandas as gpd

boundary_path = Path("../data/raw/boundaries/geoBoundaries-NGA-ADM0_simplified.geojson")

nigeria = gpd.read_file(boundary_path)

print(nigeria.crs)
nigeria.head()

EPSG:4326


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,the Federal Republic of Nigeria,NGA,39225661B33378512450279,NGA,ADM0,"POLYGON ((5.67705 13.82055, 5.63161 13.83633, ..."


In [6]:
nigeria_wgs84 = nigeria.to_crs("EPSG:4326")

In [7]:
from dem_stitcher.stitcher import stitch_dem
from pathlib import Path
import rasterio
import numpy as np
from tqdm import tqdm

dem_dir = Path("../data/raw/copernicus_dem/chunks_250m")
dem_dir.mkdir(parents=True, exist_ok=True)

nigeria_wgs84 = nigeria.to_crs("EPSG:4326")
minx, miny, maxx, maxy = nigeria_wgs84.total_bounds

# chunk size in degrees. 2 x 2 degrees is safer than full Nigeria.
chunk_size = 2.0

chunk_paths = []
chunk_id = 0

x = minx
while x < maxx:
    y = miny
    while y < maxy:
        chunk_bounds = [
            x,
            y,
            min(x + chunk_size, maxx),
            min(y + chunk_size, maxy)
        ]

        chunk_path = dem_dir / f"nigeria_dem_250m_chunk_{chunk_id:03d}.tif"

        if not chunk_path.exists():
            try:
                dem_array, dem_profile = stitch_dem(
                    bounds=chunk_bounds,
                    dem_name="glo_30",
                    dst_ellipsoidal_height=False,
                    dst_area_or_point="Area",
                    dst_resolution=0.0025,
                    n_threads_downloading=1,
                    n_threads_reproj=1
                )

                with rasterio.open(chunk_path, "w", **dem_profile) as dst:
                    dst.write(dem_array, 1)

                print(f"Saved chunk {chunk_id}: {chunk_path}")

            except Exception as e:
                print(f"Failed chunk {chunk_id}: {chunk_bounds}")
                print(e)

        else:
            print(f"Chunk already exists: {chunk_path}")

        chunk_paths.append(chunk_path)
        chunk_id += 1
        y += chunk_size

    x += chunk_size

print(f"Finished. Total chunks attempted: {chunk_id}")

Reading tile imagery: 100%|██████████| 4/4 [00:05<00:00,  1.36s/it]


Saved chunk 0: ..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_000.tif


Reading tile imagery: 100%|██████████| 9/9 [04:35<00:00, 30.64s/it]


Saved chunk 1: ..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_001.tif


Reading tile imagery: 100%|██████████| 9/9 [06:55<00:00, 46.19s/it]


...

Saved chunk 27: ..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_027.tif


Reading tile imagery: 100%|██████████| 9/9 [01:44<00:00, 11.60s/it]


Saved chunk 28: ..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_028.tif


Reading tile imagery: 100%|██████████| 6/6 [01:19<00:00, 13.22s/it]


Saved chunk 29: ..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_029.tif
Finished. Total chunks attempted: 30


In [8]:
from pathlib import Path
import rasterio
from rasterio.merge import merge
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from shapely.geometry import box

dem_chunk_dir = Path("../data/raw/copernicus_dem/chunks_250m")
dem_chunks = sorted(list(dem_chunk_dir.glob("*.tif")))

print("DEM chunks found:", len(dem_chunks))
for f in dem_chunks[:5]:
    print(f)

DEM chunks found: 30
..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_000.tif
..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_001.tif
..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_002.tif
..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_003.tif
..\data\raw\copernicus_dem\chunks_250m\nigeria_dem_250m_chunk_004.tif


In [9]:
with rasterio.open(dem_chunks[0]) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Nodata:", src.nodata)

CRS: EPSG:4326
Resolution: (0.0025, 0.0025)
Width: 800
Height: 508
Bounds: BoundingBox(left=2.6925, bottom=5.000277777777777, right=4.6925, top=6.270277777777777)
Nodata: nan


In [10]:
src_files = [rasterio.open(fp) for fp in dem_chunks]

dem_mosaic, dem_transform = merge(src_files)

dem_meta = src_files[0].meta.copy()
dem_meta.update({
    "driver": "GTiff",
    "height": dem_mosaic.shape[1],
    "width": dem_mosaic.shape[2],
    "transform": dem_transform,
    "compress": "lzw"
})

dem_mosaic_path = Path("../data/processed/nigeria_dem_250m_mosaic.tif")

with rasterio.open(dem_mosaic_path, "w", **dem_meta) as dst:
    dst.write(dem_mosaic)

for src in src_files:
    src.close()

print("DEM mosaic saved to:", dem_mosaic_path)

DEM mosaic saved to: ..\data\processed\nigeria_dem_250m_mosaic.tif


In [11]:
with rasterio.open(dem_mosaic_path) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)

CRS: EPSG:4326
Resolution: (0.0025, 0.0025)
Width: 4794
Height: 3846
Bounds: BoundingBox(left=2.6925, bottom=4.270833333333334, right=14.677499999999998, top=13.885833333333334)


In [12]:
from shapely.geometry import box

boundary_path = Path("../data/raw/boundaries/geoBoundaries-NGA-ADM0_simplified.geojson")
nigeria = gpd.read_file(boundary_path)

nigeria_3857 = nigeria.to_crs("EPSG:3857")

minx, miny, maxx, maxy = nigeria_3857.total_bounds
grid_size = 5000

cells = []
cell_ids = []
cell_id = 0

x = minx
while x < maxx:
    y = miny
    while y < maxy:
        grid_cell = box(x, y, x + grid_size, y + grid_size)

        if nigeria_3857.intersects(grid_cell).any():
            cells.append(grid_cell)
            cell_ids.append(f"NGA_5km_{cell_id:06d}")
            cell_id += 1

        y += grid_size
    x += grid_size

grid_5km = gpd.GeoDataFrame(
    {"cell_id": cell_ids},
    geometry=cells,
    crs="EPSG:3857"
)

print(len(grid_5km))
grid_5km.head()


38458


,cell_id,geometry
0,NGA_5km_000000,"POLYGON ((304740.257 710797.614, 304740.257 71..."
1,NGA_5km_000001,"POLYGON ((304740.257 715797.614, 304740.257 72..."
2,NGA_5km_000002,"POLYGON ((304740.257 720797.614, 304740.257 72..."
3,NGA_5km_000003,"POLYGON ((304740.257 725797.614, 304740.257 73..."
4,NGA_5km_000004,"POLYGON ((304740.257 730797.614, 304740.257 73..."


In [13]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

dem_mosaic_path = Path("../data/processed/nigeria_dem_250m_mosaic.tif")
dem_3857_path = Path("../data/processed/nigeria_dem_250m_3857.tif")

with rasterio.open(dem_mosaic_path) as src:
    dst_crs = "EPSG:3857"

    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "compress": "lzw"
    })

    with rasterio.open(dem_3857_path, "w", **kwargs) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

print("DEM reprojected:", dem_3857_path)

DEM reprojected: ..\data\processed\nigeria_dem_250m_3857.tif


In [15]:
sample_grid = grid_5km.sample(
    n=100,
    random_state=42
).reset_index(drop=True)

print("Sample size:", len(sample_grid))
sample_grid.head()

Sample size: 100


,cell_id,geometry
0,NGA_5km_021171,"POLYGON ((924740.257 1315797.614, 924740.257 1..."
1,NGA_5km_005893,"POLYGON ((544740.257 1460797.614, 544740.257 1..."
2,NGA_5km_032816,"POLYGON ((1299740.257 1200797.614, 1299740.257..."
3,NGA_5km_027143,"POLYGON ((1109740.257 1250797.614, 1109740.257..."
4,NGA_5km_021541,"POLYGON ((934740.257 1230797.614, 934740.257 1..."


In [16]:
from rasterstats import zonal_stats

elevation_stats = zonal_stats(
    sample_grid,
    dem_3857_path,
    stats=["mean"],
    nodata=np.nan
)

sample_grid["mean_elevation_m"] = [
    stat["mean"] for stat in elevation_stats
]

sample_grid[["cell_id", "mean_elevation_m"]].head()

,cell_id,mean_elevation_m
0,NGA_5km_021171,481.206597
1,NGA_5km_005893,245.618586
2,NGA_5km_032816,317.852286
3,NGA_5km_027143,473.102720
4,NGA_5km_021541,724.205014


In [17]:
sample_grid["mean_elevation_m"].describe()

count     100.000000
mean      333.202333
std       221.102069
min         0.106073
25%       187.314179
50%       313.969018
75%       425.109230
max      1256.327739
Name: mean_elevation_m, dtype: float64

In [18]:
sample_grid[sample_grid["mean_elevation_m"].isna()]

,cell_id,geometry,mean_elevation_m


In [19]:
from rasterstats import zonal_stats
from tqdm import tqdm

batch_dir = Path("../outputs/tables/elevation_batches")
batch_dir.mkdir(parents=True, exist_ok=True)

batch_size = 1000
batch_paths = []

for start in tqdm(range(0, len(grid_5km), batch_size)):
    end = min(start + batch_size, len(grid_5km))
    batch = grid_5km.iloc[start:end].copy()

    batch_path = batch_dir / f"elevation_batch_{start:05d}_{end:05d}.csv"

    if batch_path.exists():
        batch_paths.append(batch_path)
        continue

    stats = zonal_stats(
        batch,
        dem_3857_path,
        stats=["mean"],
        nodata=np.nan
    )

    batch["mean_elevation_m"] = [s["mean"] for s in stats]

    batch[["cell_id", "mean_elevation_m"]].to_csv(batch_path, index=False)

    batch_paths.append(batch_path)

print("Elevation batches completed.")

100%|██████████| 39/39 [05:56<00:00,  9.15s/it]

Elevation batches completed.


In [20]:
elevation_dfs = [pd.read_csv(path) for path in sorted(batch_paths)]

df_elevation = pd.concat(elevation_dfs, ignore_index=True)

df_elevation.head()

,cell_id,mean_elevation_m
0,NGA_5km_000000,4.677827
1,NGA_5km_000001,3.671904
2,NGA_5km_000002,8.152017
3,NGA_5km_000003,14.813733
4,NGA_5km_000004,32.098690


In [21]:
print("Rows:", len(df_elevation))
print("Missing elevation values:", df_elevation["mean_elevation_m"].isna().sum())

df_elevation["mean_elevation_m"].describe()

Rows: 38458
Missing elevation values: 0


count    38458.000000
mean       327.612208
std        222.543296
min          0.000000
25%        176.420344
50%        303.970014
75%        427.134115
max       1846.287616
Name: mean_elevation_m, dtype: float64

In [22]:
slope_path = Path("../data/processed/nigeria_slope_250m_3857.tif")

with rasterio.open(dem_3857_path) as src:
    dem = src.read(1).astype("float32")
    profile = src.profile.copy()
    transform = src.transform

    pixel_size_x = transform.a
    pixel_size_y = abs(transform.e)

    dem[dem == src.nodata] = np.nan

    dz_dy, dz_dx = np.gradient(dem, pixel_size_y, pixel_size_x)

    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg = np.degrees(slope_rad)

    profile.update({
        "dtype": "float32",
        "nodata": np.nan,
        "compress": "lzw"
    })

    with rasterio.open(slope_path, "w", **profile) as dst:
        dst.write(slope_deg.astype("float32"), 1)

print("Slope raster saved:", slope_path)

C:\Users\AyaChguiri\AppData\Local\Temp\ipykernel_13416\3991373268.py:4: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  dem = src.read(1).astype("float32")


Slope raster saved: ..\data\processed\nigeria_slope_250m_3857.tif


In [23]:
slope_batch_dir = Path("../outputs/tables/slope_batches")
slope_batch_dir.mkdir(parents=True, exist_ok=True)

batch_size = 1000
slope_batch_paths = []

for start in tqdm(range(0, len(grid_5km), batch_size)):
    end = min(start + batch_size, len(grid_5km))
    batch = grid_5km.iloc[start:end].copy()

    batch_path = slope_batch_dir / f"slope_batch_{start:05d}_{end:05d}.csv"

    if batch_path.exists():
        slope_batch_paths.append(batch_path)
        continue

    stats = zonal_stats(
        batch,
        slope_path,
        stats=["mean"],
        nodata=np.nan
    )

    batch["mean_slope_deg"] = [s["mean"] for s in stats]

    batch[["cell_id", "mean_slope_deg"]].to_csv(batch_path, index=False)

    slope_batch_paths.append(batch_path)

print("Slope batches completed.")

100%|██████████| 39/39 [06:00<00:00,  9.24s/it]

Slope batches completed.


In [24]:
slope_dfs = [pd.read_csv(path) for path in sorted(slope_batch_paths)]

df_slope = pd.concat(slope_dfs, ignore_index=True)

print("Rows:", len(df_slope))
print("Missing slope values:", df_slope["mean_slope_deg"].isna().sum())

df_slope["mean_slope_deg"].describe()

Rows: 38458
Missing slope values: 0


count    38458.000000
mean         1.276627
std          1.835309
min          0.000000
25%          0.403378
50%          0.812983
75%          1.306412
max         19.337632
Name: mean_slope_deg, dtype: float64

In [25]:
landcover_path = Path("../outputs/tables/nigeria_landcover_fraction_5km.csv")

df_landcover = pd.read_csv(landcover_path)

df_final = (
    df_landcover
    .merge(df_elevation, on="cell_id")
    .merge(df_slope, on="cell_id")
)

df_final.head()

,cell_id,longitude,latitude,trees,shrubland,grassland,cropland,built_up,bare_sparse_vegetation,snow_ice,permanent_water_bodies,herbaceous_wetland,mangroves,moss_lichen,fraction_sum,valid_pixel_count,mean_elevation_m,mean_slope_deg
0,NGA_5km_000000,2.71507,6.394346,0.338479,0.020117,0.201586,0.004714,0.244382,0.011917,0.0,0.091124,0.087680,0.000000,0.0,1.0,288904,4.677827,0.212004
1,NGA_5km_000001,2.71507,6.438981,0.407654,0.009360,0.134719,0.001627,0.057950,0.000177,0.0,0.167329,0.220170,0.001014,0.0,1.0,288904,3.671904,0.242650
2,NGA_5km_000002,2.71507,6.483611,0.898153,0.000843,0.037976,0.005157,0.053852,0.000541,0.0,0.000000,0.003478,0.000000,0.0,1.0,288365,8.152017,0.422796
3,NGA_5km_000003,2.71507,6.528238,0.835599,0.032848,0.045264,0.011118,0.056472,0.000000,0.0,0.000000,0.018698,0.000000,0.0,1.0,288904,14.813733,0.761646
4,NGA_5km_000004,2.71507,6.572860,0.823949,0.025523,0.037824,0.044582,0.064980,0.000003,0.0,0.000000,0.003138,0.000000,0.0,1.0,288365,32.098690,0.645393


In [26]:
print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))

df_final[["mean_elevation_m", "mean_slope_deg"]].describe()

Rows: 38458
Columns: 18


,mean_elevation_m,mean_slope_deg
count,38458.000000,38458.000000
mean,327.612208,1.276627
std,222.543296,1.835309
min,0.000000,0.000000
25%,176.420344,0.403378
50%,303.970014,0.812983
75%,427.134115,1.306412
max,1846.287616,19.337632


In [27]:
output_path = Path("../outputs/tables/deployment_site_database_v2.csv")

df_final.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ..\outputs\tables\deployment_site_database_v2.csv


In [28]:
df_final.sample(
    500,
    random_state=42
).to_csv(
    "../outputs/tables/deployment_site_database_sample.csv",
    index=False
)